In [2]:
import torch
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print(f'lenght of the characters in the dataset {len(text)}')

lenght of the characters in the dataset 1115394


In [5]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [6]:
chars = sorted(list(set(text)))
vocabulary_size = len(chars)
print('This are the single chars of the data')
print(' '.join(chars))
print(vocabulary_size)


This are the single chars of the data

   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z
65


### Encoder and Decoder tokenization

In [7]:
#Encoder
stoi = {character : index for index, character in enumerate(chars)}
encode = lambda s : [stoi[character] for character in s] # encoder: take a string, output a list of integers


#Decoder
itos = {index : character for index, character in enumerate(chars)}
decode = lambda l : ''.join([itos[index] for index in l])

In [8]:
print(encode('Hi there my name is Manuel'))
print(decode(encode('Hi there my name is Manuel')))

[20, 47, 1, 58, 46, 43, 56, 43, 1, 51, 63, 1, 52, 39, 51, 43, 1, 47, 57, 1, 25, 39, 52, 59, 43, 50]
Hi there my name is Manuel


### Now encode all the dataset in the for using torch

In [9]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:10])
print(decode(encode(text[:10])))

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])
First Citi


### Divide in training and validation

In [10]:
n_len = int(0.90 * len(data))
train_data = data[:n_len]
val_data = data[n_len:]
len(train_data), len(val_data)

(1003854, 111540)

In [11]:
sequence_lenght = 8
train_data[:sequence_lenght + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
input_x = train_data[ :sequence_lenght]
target_y = train_data[1: sequence_lenght + 1 ]

for c in range(sequence_lenght):
    input = input_x[: c + 1]
    target = target_y[c]
    print(f'This is the input {input} this is the target {target}')

This is the input tensor([18]) this is the target 47
This is the input tensor([18, 47]) this is the target 56
This is the input tensor([18, 47, 56]) this is the target 57
This is the input tensor([18, 47, 56, 57]) this is the target 58
This is the input tensor([18, 47, 56, 57, 58]) this is the target 1
This is the input tensor([18, 47, 56, 57, 58,  1]) this is the target 15
This is the input tensor([18, 47, 56, 57, 58,  1, 15]) this is the target 47
This is the input tensor([18, 47, 56, 57, 58,  1, 15, 47]) this is the target 58


In [13]:
from utils import get_batch


In [14]:
batch_size = 4 # how many independent sequences will we process in parallel?
sequence_lenght = 8 # What is the maximum context lenght for the prediction? 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

xb, yb = get_batch(
    split='train',
    batch_size=batch_size,
    sequence_length=sequence_lenght,
    train_data=train_data,
    val_data=val_data,
    device=device)


print("The inputs")
print(xb.shape)
print(xb)

print("The targets")
print(yb.shape)
print(yb)


The inputs
torch.Size([4, 8])
tensor([[53, 44,  1, 51, 63,  1, 40, 56],
        [43, 50, 50,  1, 39, 57,  0, 57],
        [52, 42,  1, 50, 43, 58,  1, 46],
        [58, 53, 47, 50,  5, 42,  1, 61]])
The targets
torch.Size([4, 8])
tensor([[44,  1, 51, 63,  1, 40, 56, 39],
        [50, 50,  1, 39, 57,  0, 57, 46],
        [42,  1, 50, 43, 58,  1, 46, 47],
        [53, 47, 50,  5, 42,  1, 61, 47]])


In [15]:
for b in range(batch_size):
    for sl in range(sequence_lenght):
        context = xb[b, :sl+1]
        target = yb[b, sl]
        print(f'This is the input {context.tolist()} this is the target {target}')
    break

This is the input [53] this is the target 44
This is the input [53, 44] this is the target 1
This is the input [53, 44, 1] this is the target 51
This is the input [53, 44, 1, 51] this is the target 63
This is the input [53, 44, 1, 51, 63] this is the target 1
This is the input [53, 44, 1, 51, 63, 1] this is the target 40
This is the input [53, 44, 1, 51, 63, 1, 40] this is the target 56
This is the input [53, 44, 1, 51, 63, 1, 40, 56] this is the target 39


In [16]:
print(f'The single input for the transformer \n {xb}')

The single input for the transformer 
 tensor([[53, 44,  1, 51, 63,  1, 40, 56],
        [43, 50, 50,  1, 39, 57,  0, 57],
        [52, 42,  1, 50, 43, 58,  1, 46],
        [58, 53, 47, 50,  5, 42,  1, 61]])


In [17]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocabulary_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.7263, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


In [18]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

/Users/administrador/Library/Python/3.11/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb =get_batch(
    split='train',
    batch_size=batch_size,
    sequence_length=sequence_lenght,
    train_data=train_data,
    val_data=val_data,
    device=device)

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


2.5727508068084717


In [20]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


Iyoteng h hasbe pave pirance
Rie hicomyonthar's
Plinseard ith henoure wounonthioneir thondy, y heltieiengerofo'dsssit ey
KIN d pe wither vouprrouthercc.
hathe; d!
My hind tt hinig t ouchos tes; st yo hind wotte grotonear 'so it t jod weancotha:
h hay.JUCle n prids, r loncave w hollular s O:
HIs; ht anjx?

DUThinqunt.

LaZAnde.
athave l.
KEONH:
ARThanco be y,-hedarwnoddy scace, tridesar, wnl'shenous s ls, theresseys
PlorseelapinghiybHen yof GLUCEN t l-t E:
I hisgothers je are!-e!
QLYotouciullle'z


### Scaled Dot-product Attention

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [21]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32 # batch, time, channels
x = torch.randn(B, T, C)


head_size = 16
queries = nn.Linear(C, head_size, bias=False) # What im looking for?
keys = nn.Linear(C, head_size, bias=False) #What do I contain?
values = nn.Linear(C, head_size, bias=False) #If you make attention to me, this is the real information I will give you

q = queries(x) # B, t, 16
k = keys(x) # B, t, 16
v = values(x) # B, t, 16
wei = (q @ k.transpose(-2, -1)) / (head_size ** 0.5) # B, T, 16 @ B, 16, T -> B, T, T


tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei[0])

output = wei @ v
output[0] #If the score is high == means a lot of attention

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5221, 0.4779, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3602, 0.3210, 0.3188, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2980, 0.4039, 0.1578, 0.1404, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1643, 0.1243, 0.1678, 0.1865, 0.3570, 0.0000, 0.0000, 0.0000],
        [0.2656, 0.2110, 0.1137, 0.1214, 0.2018, 0.0865, 0.0000, 0.0000],
        [0.1761, 0.1327, 0.1371, 0.0974, 0.1476, 0.1918, 0.1173, 0.0000],
        [0.1046, 0.1260, 0.0922, 0.0906, 0.1476, 0.1588, 0.1432, 0.1371]],
       grad_fn=<SelectBackward0>)


tensor([[-0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
         -0.8873,  0.1907,  0.1762, -0.5943, -0.4812, -0.4860,  0.2862,  0.5710],
        [ 0.3156,  0.0704, -0.0706, -0.1604, -0.1344,  0.1558, -0.2001, -0.2886,
         -0.4120,  0.4947,  0.4806, -0.3233, -0.0231, -0.0158,  0.0837,  0.9683],
        [ 0.4029, -0.0241, -0.2422,  0.0145,  0.0144, -0.0129, -0.0916, -0.1296,
         -0.3266,  0.0527,  0.3795, -0.0745, -0.1562, -0.0397, -0.0320,  1.0982],
        [ 0.4779, -0.2057, -0.2656,  0.1018,  0.0853, -0.1675, -0.1532, -0.1084,
         -0.1658,  0.2856,  0.3110, -0.0448,  0.0502,  0.1370,  0.1077,  0.9660],
        [ 0.3579,  0.2417,  0.0711,  0.1706,  0.2915,  0.1982,  0.2582,  0.1424,
         -0.3187, -0.4826, -0.1465, -0.0625, -0.4228,  0.1282,  0.0811,  0.6980],
        [ 0.2370,  0.1631, -0.0279,  0.1223,  0.1766,  0.1643,  0.0528, -0.0590,
         -0.2692, -0.0797,  0.0590, -0.0432, -0.2163,  0.0780,  0.1760,  0.6983],
        [ 0.0250,  0.1

In [22]:
output.mean(), output.var()

(tensor(0.0035, grad_fn=<MeanBackward0>),
 tensor(0.1453, grad_fn=<VarBackward0>))

In [23]:
values

Linear(in_features=32, out_features=16, bias=False)

In [24]:
from gpt import MaskedHeadAttention, MultiHeadAttention, MLP, GPTLM

attention = MaskedHeadAttention(16, 32, 8, 0.2)
output = attention.forward(x=x)
output, attention

(tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
            0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
            0.0000,  0.0000],
          [-0.2844,  0.1275,  0.1942, -0.5983,  0.3644, -0.4633,  0.4843,
           -0.6711,  0.3324,  0.3294, -0.5645, -0.3945, -0.6330, -0.6087,
            0.3471,  0.2287],
          [-0.4074,  0.0324, -0.1808,  0.2924,  0.2654, -0.6448,  0.2879,
           -0.0587,  0.6478, -0.2641, -0.1457, -0.2523, -0.6368,  0.0205,
            0.4296,  0.3099],
          [ 0.0548,  0.0462,  0.1777, -0.0864, -0.0036, -0.1651,  0.1763,
           -0.2683, -0.2354,  0.0982, -0.0303,  0.0745,  0.0335, -0.1204,
           -0.1253,  0.4279],
          [-0.0885,  0.0340,  0.1956,  0.0031,  0.1088, -0.2950,  0.2649,
           -0.1932,  0.0686,  0.0408, -0.2388, -0.0542, -0.0390, -0.2358,
            0.0968,  0.2305],
          [-0.3217,  0.1085,  0.0581,  0.2478,  0.0626, -0.5112, -0.0540,
           -0.1324,  0.6218, -0.1826

In [25]:
MultiHeadAttention(num_heads=2, head_size=16, n_embeddings=32, sequence_lenght=8, dropout=0.2)


MultiHeadAttention(
  (heads): ModuleList(
    (0-1): 2 x MaskedHeadAttention(
      (queries): Linear(in_features=32, out_features=16, bias=False)
      (keys): Linear(in_features=32, out_features=16, bias=False)
      (values): Linear(in_features=32, out_features=16, bias=False)
      (dropout): Dropout(p=0.2, inplace=False)
    )
  )
  (proj): Linear(in_features=32, out_features=32, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)

In [26]:
MLP(512, 0.2)

MLP(
  (net): Sequential(
    (0): Linear(in_features=512, out_features=2048, bias=True)
    (1): ReLU()
    (2): Linear(in_features=2048, out_features=512, bias=True)
    (3): Dropout(p=0.2, inplace=False)
  )
)

In [27]:
GPTLM(vocab_size=vocabulary_size, n_embeddings=16, sequence_lenght=8, n_heads=8, n_layers=8, dropout=0.2, device=device)

GPTLM(
  (token_embedding_table): Embedding(65, 16)
  (positional_encoding): Embedding(8, 16)
  (blocks): Sequential(
    (0): Block(
      (head_attention): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x MaskedHeadAttention(
            (queries): Linear(in_features=16, out_features=2, bias=False)
            (keys): Linear(in_features=16, out_features=2, bias=False)
            (values): Linear(in_features=16, out_features=2, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=16, out_features=16, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (fnn): MLP(
        (net): Sequential(
          (0): Linear(in_features=16, out_features=64, bias=True)
          (1): ReLU()
          (2): Linear(in_features=64, out_features=16, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
      

In [28]:
gpt_model = GPTLM(vocab_size=vocabulary_size, n_embeddings=384, sequence_lenght=256, n_heads=6, n_layers=6, dropout=0.2, device=device)
m = gpt_model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

10.788929 M parameters


### Model prediction of the dataset, GPT

In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


gpt_model.load_state_dict(
    torch.load("best_gpt_model.pt", map_location=device)
)
final_model_parameters = gpt_model.to(device)

print(sum(p.numel() for p in final_model_parameters.parameters())/1e6, 'M parameters')


context = torch.zeros((1, 1), dtype=torch.long, device=device)


print(decode(gpt_model.generate(context, max_new_tokens=1000)[0].tolist()))

10.788929 M parameters

A wifiest and your maject:
Is the justice changed may a nature,
For he eyes of consent you? Weeping grave, mumble
Is neared in breashing before for you have
The first of Martius. Chasafat, this face
Do't, adies in Edward, and Antium:
There has may new be by Antique's forth
Unto Grimio in the which state redicte conceit.

COMINIUS:
A letters: I seem in thee;
Or when most reason well-pile their enster's doles
Having been and in their present semars.
But myself and no hat our teempty,
Ray how I granted and habits may crype
The stumbling vanches; we will beat past of all,
Come her sham pay and sad ground rest in foe. Hermious were
eyes the seal vast.

LEONTES:
Upon this think-corn, tending on the lose,
I say, and take he not; indeed thee no good
That says natures make, bale; yet in sad
We are meet thousand and wout; and so, we will
Who pity befalls no perfusal natures;
in that, as may have I sworn in a necture,
The lover was true of a defence.

MENENIUS:
The most me